# 03. 다중시기 LST → 격자 집계

여름 4개 날짜(06-02·06-18·07-04·08-05, 09-22는 구름 77%+9월이라 제외)를
픽셀별 유효값으로 합성한 뒤, 30m 격자에 **픽셀 중심점 point-in-polygon 평균**으로 집계한다.

- `lst_mean`: 여름 합성 평균 LST(°C)
- `lst_aug05`: 폭염 피크(8/5) LST(°C)
- `n_pix`: 격자에 들어온 유효 픽셀 수

산출물: `data/processed/grid_features.geojson`, `output/figures/fig03_lst_grid.png`

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, geopandas as gpd, pandas as pd
from src.config import (DATA_INTERIM, DATA_PROCESSED, OUT_FIGURES, CRS_LANDSAT, CRS_SUBMIT,
                        LANDSAT_SUMMER, LANDSAT_PEAK, landsat_file)
from src.raster.landsat import lst_multitemporal, lst_zscore_composite
from src.raster.geo import read_geotransform, pixel_center_coords

grid = gpd.read_file(DATA_INTERIM / "grid_tagged.geojson").to_crs(CRS_LANDSAT)
mean_c, count = lst_multitemporal(LANDSAT_SUMMER)   # 절대 °C 합성 (물리 해석용)
lst_z = lst_zscore_composite(LANDSAT_SUMMER)        # z-score 합성 (날짜 편향 제거, 지수용)
mean_c.shape, round(float(np.nanmean(mean_c)), 1)

((1358, 1076), 36.0)

In [2]:
# 유효 픽셀 → 중심점 GeoDataFrame (LST 좌표계 32652)
gt = read_geotransform(str(landsat_file(LANDSAT_PEAK, "ST_B10")))
xs, ys = pixel_center_coords(gt["gt"], gt["width"], gt["height"])
valid = ~np.isnan(mean_c)
pts = gpd.GeoDataFrame(
    {"lst_mean": mean_c[valid], "lst_z": lst_z[valid]},
    geometry=gpd.points_from_xy(xs[valid], ys[valid]), crs=CRS_LANDSAT,
)
len(pts)

150874

In [3]:
# 픽셀점을 격자에 공간조인 → 격자별 평균
j = gpd.sjoin(pts, grid[["grid_id", "geometry"]], predicate="within", how="inner")
agg = j.groupby("grid_id").agg(lst_mean=("lst_mean", "mean"),
                               lst_z=("lst_z", "mean"),
                               n_pix=("lst_mean", "size")).reset_index()
grid_feat = grid.merge(agg, on="grid_id", how="left")
cov = grid_feat["lst_mean"].notna().sum()
print(f"LST 있는 격자 {cov:,} / 전체 {len(grid_feat):,} ({100*cov/len(grid_feat):.1f}%)")

LST 있는 격자 52,687 / 전체 54,029 (97.5%)


In [4]:
# 지구별 LST — 절대 합성 vs z-score (편향 보정 효과 확인)
print(grid_feat.groupby("zoneName")[["lst_mean", "lst_z"]].mean().round(2))
print("\n절대 합성:", grid_feat["lst_mean"].describe()[["mean", "min", "max"]].round(1).to_dict())
print("z-score  :", grid_feat["lst_z"].describe()[["mean", "min", "max"]].round(2).to_dict())

                lst_mean  lst_z
zoneName                       
고양창릉 공공주택지구    37.860001   0.45
남양주왕숙 공공주택지구   37.990002   0.55
성남분당           37.680000   0.43
성남판교지구 택지개발사업  36.970001   0.22

절대 합성: {'mean': 37.6, 'min': 28.0, 'max': 54.0}
z-score  : {'mean': 0.42, 'min': -1.84, 'max': 4.79}


In [5]:
# 저장 (EPSG:4326, 좌표 6자리로 용량 절감) + 2패널: 절대 vs z-score
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
grid_feat.to_crs(CRS_SUBMIT).to_file(
    DATA_PROCESSED / "grid_features.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)

import matplotlib.pyplot as plt
from src.viz.fonts import setup_font
setup_font()   # 한글 폰트 자동 적용(COMPAS 미보유 시 동봉 폰트 등록)
fig, axes = plt.subplots(1, 2, figsize=(15, 8))
grid_feat.plot(column="lst_mean", cmap="inferno", ax=axes[0], linewidth=0, legend=True,
               legend_kwds={"shrink": 0.6}, missing_kwds={"color": "#eeeeee"})
axes[0].set_title("여름 절대 합성 LST (°C)")
grid_feat.plot(column="lst_z", cmap="RdBu_r", ax=axes[1], linewidth=0, legend=True,
               legend_kwds={"shrink": 0.6}, vmin=-2, vmax=2, missing_kwds={"color": "#eeeeee"})
axes[1].set_title("z-score 합성 (편향 보정, 상대 열)")
for ax in axes:
    ax.set_axis_off()
plt.tight_layout(); plt.savefig(OUT_FIGURES / "fig03_lst_grid.png", dpi=130); plt.close()